

## 多头注意力机制 (Multi-Head Attention, MHA) 核心笔记

### 一、 适用场景与意义
MHA 是现代大语言模型（LLM）强大性能的基石，几乎被应用于所有基于 **Transformer架构** 的模型中（如 BERT、GPT-2/3、LLaMA 等）。

* **长距离依赖建模**：打破了位置顺序的限制，模型能够关注到序列中的任意词元（Token），有效捕捉长距离依赖关系。
* **自回归预测**：在 GPT 系列等解码器模型中，结合 **因果遮罩 (Masked MHA)**，使模型能够基于前文灵活、准确地预测下一个单词。
* **高效并行计算**：多头注意力的计算结构天然适合在 GPU 上进行加速，具备极高的并行处理效率。

---

### 二、 标准 MHA 的局限性
尽管功能强大，但标准 MHA 在大模型中存在明显的**计算和内存劣势**，特别是面对长文本和大规模推理时：

#### 1. 计算复杂度随序列长度呈二次增长
* **复杂度公式**：假设序列长度为 $N$、隐藏维度为 $d$、注意力头数为 $h$，每一步注意力计算的时间复杂度约为 $O(N^2 \cdot d)$。
* **长文本代价高昂**：由于计算量随序列长度 $N$ 呈**二次方增长**，导致模型在处理长文本时计算开销极大。

#### 2. KV 缓存 (KV Cache) 导致显存占用惊人
* **自回归生成的内存压力**：在聊天模型逐字生成文本的过程中，为了避免重复计算，Transformer 会使用 **KV 缓存** 来保存过去所有标记的键 (Key) 和值 (Value)。
* **线性增长的存储开销**：在标准 MHA 中，如果每个注意力头都保留一份专属的键和值，KV 缓存的大小将随着注意力头数呈**线性增长**。
* **推理瓶颈**：当模型拥有数十个注意力头，且上下文窗口扩展至数千或数万时，KV 缓存会占用海量的显存/内存，直接成为拖慢**生成速度**和**大批量 (Batch) 推理**的核心瓶颈。

---

In [ ]:
# import torch
# import torch.nn as nn
# import math


# class MultiHeadAttention(nn.Module):
#     def __init__(self, d_model, num_heads):
#         super().__init__()
#         self.d_model = d_model
#         self.num_heads = num_heads
#         self.d_k = d_model // num_heads

#         self.Wq = nn.Linear(d_model, d_model)
#         self.Wk = nn.Linear(d_model, d_model)
#         self.Wv = nn.Linear(d_model, d_model)
#         self.Wo = nn.Linear(d_model, d_model)

#     def forward(self, Q, K, V, mask=None):
#         batch_size = Q.size(0)

#         Q = self.Wq(Q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
#         K = self.Wk(K).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
#         V = self.Wv(V).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

#         scores = torch.matmul(Q, K.transpose(-1, -2)) / math.sqrt(self.d_k)

#         if mask is not None:
#             scores = scores.masked_fill(mask == 0, -1e9) # 新增这行

#         attn_weights = torch.softmax(scores, -1)

#         output = (
#             torch.matmul(attn_weights, V)  #
#             .transpose(1, 2)
#             .contiguous()
#             .view(batch_size, -1, self.num_heads * self.d_k)
#         )

#         print(output.shape)

#         output = self.Wo(output)

#         return output, attn_weights


# def test_multi_head_attention():
#     # 参数设置
#     batch_size = 2
#     seq_len = 5
#     d_model = 512
#     num_heads = 8

#     device = "cuda"

#     # 初始化模型
#     mha = MultiHeadAttention(d_model, num_heads).to(device)

#     # 模拟输入数据 (Batch, Seq_Len, D_Model)
#     x = torch.randn(batch_size, seq_len, d_model).to(device)

#     # 前向传播
#     try:
#         out, weights = mha(x, x, x)
#         print("✅ 测试成功！")
#         print(f"输入形状: {x.shape}")
#         print(f"输出形状: {out.shape} (应为 [2, 5, 512])")
#         print(f"权重形状: {weights.shape} (应为 [2, 8, 5, 5])")
#     except Exception as e:
#         print(f"❌ 测试失败: {e}")


# test_multi_head_attention()


torch.Size([2, 5, 512])
✅ 测试成功！
输入形状: torch.Size([2, 5, 512])
输出形状: torch.Size([2, 5, 512]) (应为 [2, 5, 512])
权重形状: torch.Size([2, 8, 5, 5]) (应为 [2, 8, 5, 5])


In [9]:
import torch
import torch.nn as nn
import math


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.Wo = nn.Linear(d_model, d_model)

    def forward(self, Q, K, V, mask=None):
        batch_size = Q.size(0)

        Q = self.Wq(Q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.Wk(K).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.Wv(V).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-1, -2)) / math.sqrt(self.d_k)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        attn_weights = torch.softmax(scores, -1)



        output = (
            torch.matmul(attn_weights, V)
            .transpose(1, 2)
            .contiguous()
            .view(batch_size, -1, self.d_k * self.num_heads)
        )

        output = self.Wo(output)
        return output,  attn_weights

def test_multi_head_attention():
    # 参数设置
    batch_size = 2
    seq_len = 5
    d_model = 512
    num_heads = 8

    # 自动检测设备 (防止没有 GPU 时报错)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # 初始化模型
    mha = MultiHeadAttention(d_model, num_heads).to(device)

    # 模拟输入数据 (Batch, Seq_Len, D_Model)
    x = torch.randn(batch_size, seq_len, d_model).to(device)

    # 构造一个因果掩码 (Causal Mask)
    # 下三角矩阵（包含对角线），1代表可见，0代表不可见（遮蔽）
    # 扩展维度为 (1, 1, seq_len, seq_len)，利用广播机制适配 batch 和 num_heads
    mask = (
        torch.tril(torch.ones(seq_len, seq_len)).view(1, 1, seq_len, seq_len).to(device)
    )

    # 前向传播
    try:
        out, weights = mha(x, x, x, mask=mask)
        print("✅ 测试成功！\n" + "-" * 30)
        print(f"输入形状: {x.shape}")
        print(f"输出形状: {out.shape} (应为 [{batch_size}, {seq_len}, {d_model}])")
        print(
            f"权重形状: {weights.shape} (应为 [{batch_size}, {num_heads}, {seq_len}, {seq_len}])"
        )

        print("\n🔍 检查注意力权重 (以第一个Batch的第一个Head为例):")
        # 打印出的矩阵右上角应该全为0，表示未来的信息被成功 Mask 掉了
        print(torch.round(weights[0, 0] * 1000) / 1000)

    except Exception as e:
        print(f"❌ 测试失败: {e}")


test_multi_head_attention()

✅ 测试成功！
------------------------------
输入形状: torch.Size([2, 5, 512])
输出形状: torch.Size([2, 5, 512]) (应为 [2, 5, 512])
权重形状: torch.Size([2, 8, 5, 5]) (应为 [2, 8, 5, 5])

🔍 检查注意力权重 (以第一个Batch的第一个Head为例):
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3240, 0.6760, 0.0000, 0.0000, 0.0000],
        [0.3510, 0.3570, 0.2920, 0.0000, 0.0000],
        [0.2550, 0.4240, 0.2030, 0.1180, 0.0000],
        [0.1950, 0.1460, 0.2280, 0.2870, 0.1440]], device='cuda:0',
       grad_fn=<DivBackward0>)
